In [ ]:
# Run first: locate the repository root from any notebook subfolder.
from pathlib import Path
import os
import sys
project_root = next((candidate for candidate in [Path.cwd(), *Path.cwd().parents]
                     if (candidate / "app.py").is_file()
                     and (candidate / "retrieval_final.py").is_file()), None)
if project_root is None:
    raise RuntimeError("Start Jupyter within the project repository.")
os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))


In [ ]:
# Configure the Qwen LoRA training environment, dataset, output directory, and device.
import os

os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
os.environ["HF_ENABLE_PARALLEL_LOADING"] = "false"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import pandas as pd
import torch

from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"
TRAINING_FILE = "data/enron_qa_training_2000.csv"
OUTPUT_DIR = "qwen_enron_lora"

DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
print("Device:", DEVICE)

In [ ]:
# Load and inspect the 2,000 Enron question-answer examples used for LoRA fine-tuning.
training_df = pd.read_csv(TRAINING_FILE)

print("Rows:", len(training_df))
print(training_df.columns.tolist())

display(training_df.head())

In [ ]:
# Define the grounded answer-and-source format used for LoRA supervised fine-tuning.
SYSTEM_PROMPT = """
Use the supplied email evidence to answer the question.

Do not use outside knowledge or invent information.

Return:
ANSWER: <answer>
SOURCE: <email_id>
""".strip()

# Convert each training row into the prompt and completion format expected by SFTTrainer.
def make_example(row):
    user_content = (
        f"{SYSTEM_PROMPT}\n\n"
        f"QUESTION:\n{row['question']}\n\n"
        f"EMAIL EVIDENCE:\n"
        f"EMAIL_ID: {row['email_id']}\n"
        f"{row['evidence']}"
    )

    assistant_content = (
        f"ANSWER: {row['answer']}\n"
        f"SOURCE: {row['email_id']}"
    )

    return {
        "prompt": [
            {
                "role": "user",
                "content": user_content
            }
        ],
        "completion": [
            {
                "role": "assistant",
                "content": assistant_content
            }
        ],
    }


examples = [
    make_example(row)
    for _, row in training_df.iterrows()
]

dataset = Dataset.from_list(examples)

print(dataset[0])

In [ ]:
# Split the LoRA dataset into reproducible training and validation sets.
split_dataset = dataset.train_test_split(
    test_size=200,
    seed=42,
)

train_dataset = split_dataset["train"]
valid_dataset = split_dataset["test"]

print("Training examples:", len(train_dataset))
print("Validation examples:", len(valid_dataset))

In [ ]:
# Load the base Qwen model and tokenizer in a memory-conscious configuration for LoRA training.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16,          # replaces deprecated torch_dtype
    attn_implementation="eager",
    low_cpu_mem_usage=True,
    # Do not pass device_map here.
)

if DEVICE == "mps":
    model = model.to("mps")

model.config.use_cache = False

print("Base Qwen loaded:", next(model.parameters()).device)

In [ ]:
# Configure LoRA adapters for the attention projection layers of the base Qwen model.
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
    ],
)

print("LoRA configuration ready.")

In [ ]:
# Configure the supervised LoRA training schedule, memory settings, evaluation, and checkpointing.
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,

    num_train_epochs=2,

    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,

    learning_rate=1e-4,

    max_length=2048,

    logging_steps=10,

    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,

    completion_only_loss=True,

    fp16=True,

    report_to="none",

    seed=42,
)

In [ ]:
# Initialise the supervised fine-tuning trainer with Qwen, LoRA, and the prepared datasets.
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    peft_config=lora_config,
    processing_class=tokenizer,
)

print("Trainer ready.")

In [ ]:
# Confirm how many Qwen parameters are trainable through the LoRA adapters.
trainer.model.print_trainable_parameters()

In [ ]:
# Train the LoRA adapters and record the total fine-tuning time.
import time

start_time = time.time()

train_result = trainer.train()

training_seconds = time.time() - start_time

print()
print(f"Full LoRA training time: {training_seconds:.1f} seconds")
print(f"Full LoRA training time: {training_seconds / 60:.2f} minutes")
print(f"Full LoRA training time: {training_seconds / 3600:.2f} hours")

In [ ]:
# Save the trained LoRA adapter and tokenizer for later evaluation and inference.
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("LoRA adapter saved to:", OUTPUT_DIR)

In [ ]:

print("Saved files:")
for file in os.listdir(OUTPUT_DIR):
    print(file)

In [ ]:
# Configure LoRA generation evaluation using the same retrieval pipeline and 50-question benchmark as base Qwen.
import json
import subprocess
import sys
import time
from pathlib import Path

import pandas as pd
import torch

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

EVALUATION_FILE = "data/enron_evaluation_250.csv"
RETRIEVAL_NOTEBOOK = "notebooks/03_retrieval/Retrieval-Final.ipynb"

BASE_MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"
LORA_DIR = "qwen_enron_lora"

RETRIEVAL_K = 10
MAX_INPUT_TOKENS = 8192
MAX_NEW_TOKENS = 256
MAX_GENERATION_SECONDS = 60

RETRIEVAL_RESULTS_FILE = Path(
    "generation_retrieval_lora_50.json"
)

QUESTION_NUMBERS_TO_RUN = [
    12, 20, 24, 25, 30, 36, 38, 43, 44, 49,
    50, 60, 68, 69, 74, 79, 80, 91, 94, 98,
    100, 104, 113, 122, 130, 142, 157, 166, 190, 224,

    1, 8, 14, 18, 32, 45, 55, 64, 82,
    95, 110, 121, 144, 169, 188, 219, 237,

    4, 41, 102
]

DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"

print("Device:", DEVICE)
print("Questions:", len(QUESTION_NUMBERS_TO_RUN))

In [ ]:
# Load the evaluation set and select the same 50 questions used for the base Qwen benchmark.
questions = pd.read_csv(EVALUATION_FILE).reset_index(drop=True)

questions.insert(
    0,
    "question_number",
    range(1, len(questions) + 1)
)

questions = (
    questions
    .set_index("question_number")
    .loc[QUESTION_NUMBERS_TO_RUN]
    .reset_index()
)

print("Questions loaded:", len(questions))

display(
    questions[
        [
            "question_number",
            "question",
            "answer",
            "email_id",
        ]
    ].head()
)

In [ ]:
# Run final retrieval for all 50 LoRA evaluation questions in a separate process.
# Save the retrieved evidence before loading the generation model.
questions_to_retrieve = questions[
    ["question_number", "question"]
].to_dict("records")

retrieval_code = f"""
import json

RUN_RETRIEVAL_EXAMPLE = False

get_ipython().run_line_magic(
    'run',
    '-i "{RETRIEVAL_NOTEBOOK}"'
)

questions = {questions_to_retrieve!r}
results = []

for row in questions:
    chunks = retrieve_chunks(
        row["question"],
        k={RETRIEVAL_K}
    )

    results.append(chunks)

    print(
        f"Retrieved question {{row['question_number']}}",
        flush=True
    )

with open(
    r"{RETRIEVAL_RESULTS_FILE}",
    "w"
) as f:
    json.dump(results, f)
"""

process = subprocess.run(
    [
        sys.executable,
        "-m",
        "IPython",
        "-c",
        retrieval_code,
    ],
    capture_output=True,
    text=True,
)

print(process.stdout)

if process.returncode != 0:
    print(process.stderr)
    raise RuntimeError("Retrieval failed.")

In [ ]:
# Load the saved retrieval results and verify that evidence was retrieved for all evaluation questions.
with open(RETRIEVAL_RESULTS_FILE) as f:
    retrieval_results = json.load(f)

print(
    f"Retrieved evidence for "
    f"{len(retrieval_results)} questions."
)

print(
    "Chunks for first question:",
    len(retrieval_results[0])
)

In [ ]:
# Define the same grounded generation instructions used for the base Qwen evaluation.
UNKNOWN_RESPONSE = (
    "I could not find that information "
    "in the provided emails."
)

SYSTEM_PROMPT = f"""
Use the supplied email evidence to answer the question.
You may combine information across multiple pieces of evidence.

Do not use outside knowledge or invent information.

If the answer cannot be determined from the supplied evidence, say:
{UNKNOWN_RESPONSE}

Return:
ANSWER: <answer>
SOURCE: <email_id>
""".strip()

# Format the question and retrieved email chunks into the LoRA evaluation prompt.
def build_prompt(question, chunks):
    evidence_blocks = []

    for chunk in chunks:
        evidence_blocks.append(
            f"EMAIL_ID: {chunk['email_id']}\n"
            f"SUBJECT: {chunk.get('subject', '')}\n"
            f"DATE: {chunk.get('date', '')}\n"
            f"EVIDENCE:\n"
            f"{chunk['chunk_text']}"
        )

    evidence = "\n\n---\n\n".join(
        evidence_blocks
    )

    return (
        f"{SYSTEM_PROMPT}\n\n"
        f"QUESTION:\n{question}\n\n"
        f"EMAIL EVIDENCE:\n{evidence}"
    )

In [ ]:
# Load base Qwen and attach the trained Enron LoRA adapter for generation evaluation.
tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL_NAME
)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    dtype=torch.float16,
    attn_implementation="eager",
    low_cpu_mem_usage=True,
)

if DEVICE == "mps":
    base_model = base_model.to("mps")

model = PeftModel.from_pretrained(
    base_model,
    LORA_DIR
)

model.eval()
model.config.use_cache = True

print("LoRA Qwen loaded successfully.")

In [ ]:
row = questions.iloc[0]
chunks = retrieval_results[0]

prompt = build_prompt(
    row["question"],
    chunks
)

messages = [
    {
        "role": "user",
        "content": prompt,
    }
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt",
    truncation=True,
    max_length=MAX_INPUT_TOKENS,
)

inputs = {
    key: value.to(DEVICE)
    for key, value in inputs.items()
}

with torch.inference_mode():
    output = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        max_time=MAX_GENERATION_SECONDS,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

generated_tokens = output[
    0,
    inputs["input_ids"].shape[1]:
]

response = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True,
).strip()

print("QUESTION:")
print(row["question"])

print("\nEXPECTED:")
print(row["answer"])

print("\nLORA RESPONSE:")
print(response)

In [ ]:
# Generate LoRA responses for all 50 questions and record their retrieval context and generation time.
# Save progress after every question so completed results survive an interrupted run.v
RESULTS_FILE = (
    "results/generation_results_lora_qwen_50.csv"
)

results = []

for i, (row, chunks) in enumerate(
    zip(
        questions.itertuples(index=False),
        retrieval_results,
    ),
    start=1,
):
    start_time = time.time()

    prompt = build_prompt(
        row.question,
        chunks,
    )

    messages = [
        {
            "role": "user",
            "content": prompt,
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_INPUT_TOKENS,
    )

    inputs = {
        key: value.to(DEVICE)
        for key, value in inputs.items()
    }

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            max_time=MAX_GENERATION_SECONDS,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated_tokens = output[
        0,
        inputs["input_ids"].shape[1]:
    ]

    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True,
    ).strip()

    generation_seconds = (
        time.time() - start_time
    )

    retrieved_ids = [
        str(chunk["email_id"])
        for chunk in chunks
    ]

    gold_email_id = str(row.email_id)

    if gold_email_id in retrieved_ids:
        gold_email_rank = (
            retrieved_ids.index(
                gold_email_id
            ) + 1
        )
    else:
        gold_email_rank = None

    results.append({
        "question_number":
            row.question_number,

        "question":
            row.question,

        "expected_answer":
            row.answer,

        "expected_email_id":
            gold_email_id,

        "difficulty":
            row.difficulty,

        "expected_email_retrieved":
            gold_email_id in retrieved_ids,

        "gold_email_rank":
            gold_email_rank,

        "model_response":
            response,

        "retrieved_email_ids":
            " | ".join(retrieved_ids),

        "generation_seconds":
            generation_seconds,
    })

    pd.DataFrame(results).to_csv(
        RESULTS_FILE,
        index=False,
    )

    print(
        f"{i}/{len(questions)} "
        f"completed - "
        f"Question {row.question_number} "
        f"({generation_seconds:.1f}s)"
    )

In [ ]:
generation_results = pd.read_csv(
    "results/generation_results_lora_qwen_50.csv"
)

In [ ]:
# Reload the completed LoRA generation results for manual inspection.
for _, row in generation_results.iterrows():
    print("=" * 80)
    print(f"QUESTION {row['question_number']}")

    print("\nQuestion:")
    print(row["question"])

    print("\nExpected answer:")
    print(row["expected_answer"])

    print("\nGold email retrieved:")
    print(row["expected_email_retrieved"])

    print("\nGold email rank:")
    print(row["gold_email_rank"])

    print("\nLoRA response:")
    print(row["model_response"])

    print("\nRetrieved email IDs:")
    print(row["retrieved_email_ids"])

    print("\nGeneration seconds:")
    print(row["generation_seconds"])

    if "generation_time_limit_reached" in generation_results.columns:
        print("\nTime limit reached:")
        print(row["generation_time_limit_reached"])

    if "generated_token_count" in generation_results.columns:
        print("\nGenerated tokens:")
        print(row["generated_token_count"])

    print()

## Binary answer accuracy comparison

An AI judge assigns 1 when the generated answer communicates every fact requested by the question and 0 when it is incorrect, missing, refuses to answer, or omits a required fact. Differences in capitalisation, punctuation, formatting, and equivalent wording do not affect correctness. The same labels and denominators are used for the base and LoRA models.

In [ ]:
# Compare base and LoRA answer accuracy overall and when the labelled source email was retrieved.
base_generation_results = pd.read_csv(
    "results/generation_results_base_qwen.csv"
)

lora_generation_results = pd.read_csv(
    "results/generation_results_lora_qwen_50.csv"
)

# Summarise answer accuracy for one model across the two evaluation scopes.
def summarise_answer_accuracy(results, model_name):
    gold_retrieved = results[
        results["expected_email_retrieved"].astype(bool)
    ]

    rows = [
        {
            "model": model_name,
            "scope": "All 50 questions",
            "correct_answers": int(results["ai_judge_score"].sum()),
            "questions": len(results),
        },
        {
            "model": model_name,
            "scope": "Gold email retrieved",
            "correct_answers": int(gold_retrieved["ai_judge_score"].sum()),
            "questions": len(gold_retrieved),
        },
    ]

    summary = pd.DataFrame(rows)
    summary["accuracy"] = (
        summary["correct_answers"]
        / summary["questions"]
    )
    return summary

answer_accuracy_comparison = pd.concat(
    [
        summarise_answer_accuracy(base_generation_results, "Base Qwen"),
        summarise_answer_accuracy(lora_generation_results, "LoRA Qwen"),
    ],
    ignore_index=True,
)

display(
    answer_accuracy_comparison.style.format({"accuracy": "{:.1%}"})
)


## Citation quality comparison

Citation quality receives 1 when the model returns a correctly formatted source email ID that appears in the retrieved evidence supplied to the model, and 0 when the citation is missing, malformed, or not among the retrieved email IDs. The same rule is applied to Base Qwen and LoRA Qwen.

In [ ]:
# Compare valid source-citation rates between base Qwen and LoRA Qwen.
base_generation_results = pd.read_csv(
    "results/generation_results_base_qwen.csv"
)

lora_generation_results = pd.read_csv(
    "results/generation_results_lora_qwen_50.csv"
)

citation_quality_comparison = pd.DataFrame({
    "model": ["Base Qwen", "LoRA Qwen"],
    "valid_citations": [
        int(base_generation_results["citation_quality_score"].sum()),
        int(lora_generation_results["citation_quality_score"].sum()),
    ],
    "questions": [
        len(base_generation_results),
        len(lora_generation_results),
    ],
})

citation_quality_comparison["citation_quality"] = (
    citation_quality_comparison["valid_citations"]
    / citation_quality_comparison["questions"]
)

display(
    citation_quality_comparison.style.format({
        "citation_quality": "{:.1%}"
    })
)


## AI faithfulness and three-metric comparison

AI faithfulness uses the same question-aware rule for both models: every factual claim in an answer must be supported by its cited retrieved email evidence. Responses without a valid citation are excluded from this metric. The grouped chart compares Base Qwen and LoRA Qwen using identical definitions and denominators for each metric.


In [ ]:
# Compare the final answer accuracy, citation quality, and faithfulness of base and LoRA Qwen.
# Report the LoRA improvement and visualise both models across the three generation metrics.
import matplotlib.pyplot as plt
import numpy as np

base_generation_results = pd.read_csv(
    "results/generation_results_base_qwen.csv"
)
lora_generation_results = pd.read_csv(
    "results/generation_results_lora_qwen_50.csv"
)

def three_metric_scores(results):
    faithfulness_scores = results["ai_faithfulness_score"].dropna()
    return {
        "Answer accuracy": results["ai_judge_score"].mean(),
        "Citation quality": results["citation_quality_score"].mean(),
        "AI faithfulness": faithfulness_scores.mean(),
    }

base_scores = three_metric_scores(base_generation_results)
lora_scores = three_metric_scores(lora_generation_results)
metric_order = [
    "Answer accuracy",
    "Citation quality",
    "AI faithfulness",
]

for metric in metric_order:
    base_value = base_scores[metric] * 100
    lora_value = lora_scores[metric] * 100
    difference = lora_value - base_value
    print(metric)
    print(f"  LoRA Qwen: {lora_value:.1f}%")
    print(f"  Base Qwen: {base_value:.1f}%")
    print(f"  Difference: {difference:+.1f} percentage points")

x = np.arange(len(metric_order))
width = 0.34
fig, ax = plt.subplots(figsize=(9.2, 5.0))
base_bars = ax.bar(
    x - width / 2,
    [base_scores[m] * 100 for m in metric_order],
    width,
    label="Base Qwen",
    color="#4C78A8",
)
lora_bars = ax.bar(
    x + width / 2,
    [lora_scores[m] * 100 for m in metric_order],
    width,
    label="LoRA Qwen",
    color="#E45756",
)

ax.bar_label(base_bars, fmt="%.1f%%", padding=3, fontsize=10)
ax.bar_label(lora_bars, fmt="%.1f%%", padding=3, fontsize=10)
ax.set_xticks(x, metric_order)
ax.set_title("Generation evaluation: Base vs LoRA")
ax.set_ylabel("Score (%)")
ax.set_ylim(0, 105)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", alpha=0.2)
ax.legend(title="")
plt.tight_layout()
plt.show()
